In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from dataset import get_dataloader
from models import LensPredictorNetwork
import os 

In [8]:
batch_size = 64
epochs = 30
learning_rate = 0.001
decay = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_image_folder = r"D:\Recent Advances"

# --- Initialization ---
print("Loading Data...")
train_loader, val_loader = get_dataloader('..\data\lens_train.csv', '..\data\lens_val.csv',base_img_dir=my_image_folder, batch_size=batch_size)

print("Initializing Model...")
model = LensPredictorNetwork(input_dim=40, output_dim=27).to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=learning_rate,weight_decay=decay)

# Tracking the best model
best_val_accuracy = 0.0

print("Starting Training...")
# --- Training Loop ---
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_idx, (light_vectors, targets) in enumerate(train_loader):
        light_vectors, targets = light_vectors.to(device), targets.to(device)
        
        # 1. Zero the gradients
        optimizer.zero_grad()
        
        # 2. Forward pass: Predict the 27 parameters
        outputs = model(light_vectors)
        
        # 3. Calculate Loss
        loss = criterion(outputs, targets)
        
        # 4. Backward pass (calculate gradients)
        loss.backward()
        
        # 5. Optimize (update weights)
        optimizer.step()
        
        # --- Metrics Tracking ---
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += targets.size(0)
        correct_train += (predicted == targets).sum().item()
        
    train_accuracy = 100 * correct_train / total_train
    
    # --- Validation Loop (After every epoch) ---
    model.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    
    with torch.no_grad():
        for light_vectors, targets in val_loader:
            light_vectors, targets = light_vectors.to(device), targets.to(device)
            outputs = model(light_vectors)
            loss = criterion(outputs, targets)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += targets.size(0)
            correct_val += (predicted == targets).sum().item()
            
    val_accuracy = 100 * correct_val / total_val
    
    print(f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_accuracy:.2f}%")
            
    # --- Save the best model ---
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        os.makedirs('saved_models', exist_ok=True)
        torch.save(model.state_dict(), 'saved_models/best_lens_predictor.pth')
        print("--> New Best Model Saved!")

<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Jerry\AppData\Local\Temp\ipykernel_15008\3225871099.py:10: SyntaxWarning: invalid escape sequence '\d'
  train_loader, val_loader = get_dataloader('..\data\lens_train.csv', '..\data\lens_val.csv',base_img_dir=my_image_folder, batch_size=batch_size)
C:\Users\Jerry\AppData\Local\Temp\ipykernel_15008\3225871099.py:10: SyntaxWarning: invalid escape sequence '\d'
  train_loader, val_loader = get_dataloader('..\data\lens_train.csv', '..\data\lens_val.csv',base_img_dir=my_image_folder, batch_size=batch_size)


Loading Data...
Initializing Model...
Starting Training...
Epoch [1/30] Train Loss: 3.1672 | Train Acc: 10.07% | Val Loss: 2.9946 | Val Acc: 10.33%
--> New Best Model Saved!
Epoch [2/30] Train Loss: 2.8849 | Train Acc: 13.93% | Val Loss: 2.8368 | Val Acc: 12.67%
--> New Best Model Saved!
Epoch [3/30] Train Loss: 2.8168 | Train Acc: 14.13% | Val Loss: 2.7893 | Val Acc: 13.33%
--> New Best Model Saved!
Epoch [4/30] Train Loss: 2.7847 | Train Acc: 14.76% | Val Loss: 2.7844 | Val Acc: 14.67%
--> New Best Model Saved!
Epoch [5/30] Train Loss: 2.7738 | Train Acc: 14.54% | Val Loss: 2.7908 | Val Acc: 13.33%
Epoch [6/30] Train Loss: 2.7689 | Train Acc: 15.30% | Val Loss: 2.7715 | Val Acc: 14.00%
Epoch [7/30] Train Loss: 2.7541 | Train Acc: 15.98% | Val Loss: 2.7716 | Val Acc: 12.67%
Epoch [8/30] Train Loss: 2.7499 | Train Acc: 15.07% | Val Loss: 2.7752 | Val Acc: 13.33%
Epoch [9/30] Train Loss: 2.7501 | Train Acc: 15.83% | Val Loss: 2.7658 | Val Acc: 14.00%
Epoch [10/30] Train Loss: 2.7448 | T